# PrivacyGate region-v6 training (Colab GPU)

Trains `region-v6-2ep-b8` with the same settings as the Mac command. Synthetic data only.
Build the bundle on the Mac first (`python -m privacygate.training.colab_bundle --out colab-bundle-v6.zip`)
and upload it to Google Drive. Run the cells top to bottom.

In [ ]:
# [cell:gpu] A GPU runtime is required (Runtime > Change runtime type > GPU).
import subprocess, torch
assert torch.cuda.is_available(), "no CUDA GPU in this runtime"
print(torch.cuda.get_device_name(0), "torch", torch.__version__)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

In [ ]:
# [cell:drive] Mount Google Drive; the bundle is read from it and all outputs are written to it.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# [cell:config] Paths. Free Colab sessions can disconnect and the trainer has no mid-epoch
# checkpoint, so the run directory and the final zip live on Drive, never only on /content.
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/privacygate")
BUNDLE = DRIVE / "colab-bundle-v6.zip"
WORK = Path("/content/privacygate")          # unpacked code + data (fast local disk)
OUT_DIR = DRIVE / "models"                   # trainer --out-dir (on Drive)
HF_HOME = Path("/content/hf-cache")          # base-model download cache
RUN = "region-v6-2ep-b8"
EPOCHS, BATCH_SIZE = 2, 8                    # same as the Mac command

In [ ]:
# [cell:unpack] Fresh unpack of the bundle.
import shutil, zipfile
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as archive:
    names = archive.namelist()
    assert not any(n.startswith(("/", "..")) or "/../" in n for n in names), "unsafe archive path"
    archive.extractall(WORK)
print(len(names), "files unpacked")

In [ ]:
# [cell:verify] Every file must match SHA256SUMS; the data must match the train-v6 manifest.
import hashlib, json
listed = {}
for line in (WORK / "SHA256SUMS").read_text().splitlines():
    digest, name = line.split("  ", 1)
    listed[name] = digest
    assert hashlib.sha256((WORK / name).read_bytes()).hexdigest() == digest, "sha256 mismatch: " + name
manifest = json.loads((WORK / "data/manifests/train-v6.json").read_text())
for split, name in (("train", "train-v6.jsonl"), ("dev", "dev-v6-ext.jsonl")):
    assert listed["data/local/augmentation/" + name] == manifest["outputs"][split]["sha256"], split
print(len(listed), "files verified")

In [ ]:
# [cell:pip] Install the pinned training libraries Colab lacks or has at another version.
# Colab's CUDA torch build and numpy are kept (reinstalling them needs a runtime restart);
# their versions are printed so they can be compared with requirements-train.txt.
import subprocess, sys
from importlib import metadata
pins = dict(line.strip().split("==") for line in (WORK / "requirements-train.txt").read_text().splitlines() if "==" in line)
KEEP = {"torch", "numpy", "pyarrow"}  # pyarrow is only needed for train-v6 generation
install = []
for name, version in pins.items():
    try:
        have = metadata.version(name)
    except metadata.PackageNotFoundError:
        have = None
    if name in KEEP:
        print(f"keep {name} {have} (pin {version})")
    elif have != version:
        install.append(f"{name}=={version}")
if install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install], check=True)
print("installed:", install or "nothing")

In [ ]:
# [cell:base_model] Download the pinned base model the trainer uses (only network step).
import os, sys
sys.path.insert(0, str(WORK))
from privacygate import mbert_data as md
os.environ["HF_HOME"] = str(HF_HOME)
os.environ.pop("HF_HUB_OFFLINE", None)
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id=md.MODEL_ID, revision=md.MODEL_REVISION,
                         allow_patterns=["config.json", "model.safetensors", "tokenizer.json",
                                         "tokenizer_config.json", "vocab.txt", "special_tokens_map.json"])
print(md.MODEL_ID, md.MODEL_REVISION, "->", path)

In [ ]:
# [cell:train] Same command as on the Mac, output directory on Drive. If the session drops,
# the partial run directory OUT_DIR/RUN is refused on restart (run_checkpoint_incomplete):
# delete or rename it on Drive and run this cell again; training restarts from epoch 1.
import os, subprocess, sys
env = {**os.environ, "HF_HOME": str(HF_HOME), "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
       "PYTHONDONTWRITEBYTECODE": "1"}
env.pop("PYTHONPATH", None)
command = [sys.executable, "-m", "privacygate.training.train_mbert", "--run", RUN,
           "--region-train-file", "data/local/augmentation/train-v6.jsonl",
           "--region-dev-file", "data/local/augmentation/dev-v6-ext.jsonl",
           "--region-manifest", "data/manifests/train-v6.json",
           "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--out-dir", str(OUT_DIR)]
process = subprocess.Popen(command, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
assert process.wait() == 0, "trainer exited with {}".format(process.returncode)

In [ ]:
# [cell:metrics] The trainer's metrics.json (aggregate numbers only).
import json
metrics = json.loads((OUT_DIR / RUN / "metrics.json").read_text())
print(json.dumps({k: metrics[k] for k in ("run", "selected_epoch", "device", "gpu_name")}, indent=1))
print(json.dumps({"dev_overall": metrics["dev"]["overall"], "character": metrics["dev"]["character"]}, indent=1))

In [ ]:
# [cell:export] Zip the trained model folder to Drive for download to the Mac.
import shutil
archive = shutil.make_archive(str(DRIVE / RUN), "zip", root_dir=OUT_DIR, base_dir=RUN)
print(archive)